In [ ]:
import numpy as np
from qubicpack.qubicfp import qubicfp
import sys,os
import glob
import yaml
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab

In [ ]:
day = '2025-05-28'
data_dir = '/home/qubic/Calib-TD/'+day+'/'
words = ['skydip']
keywords = ['*{}*'.format(word) for word in words]
for keyword in keywords:
    dirs = np.sort(glob.glob(data_dir+keyword))
    print(dirs)

In [ ]:
data_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/data/Flux_jumps/"
soft_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/lib/Calibration/"
dict_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/dicts/"
sys.path.append(os.path.abspath(soft_path))

In [ ]:
import Qfluxjumps as jr
import Qsaturation as qsat

In [ ]:
dirs[4]

The previous ones are not analyzed, since they contain very few data samples 

## 15.36.55 sky dip

In [ ]:
dataset0 = dirs[4]
a4 = qubicfp()
a4.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a4.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
thr = 2e5  # Multiple thresholds for better detection
window_size = 600  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
# mal tes 33, 70, 192

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2e5, tol=1e2, depth=False, depth_number=4)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

I cannot eliminate the TES 34, even changing a lot the parameters. Maybe it could be eliminated if this tes is separated of the others ones

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=False, use_corrected=True
)


In [ ]:
for i in TES_with_dt_jumps:
    print("TES:", i, np.mean(np.abs(metrics_haar[i]['z_scores']))/np.mean(np.abs(metrics_dt[i]['z_scores'])))

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
plt.plot(tt, corrected_tod[234], label='with DT')
plt.plot(tt, corrected_tod_nodt[234], label='without DT')
plt.plot(tt, todarray[234])

plt.legend()

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=False)

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=True)

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2805_15.09.12", dataset_name="2805_15.09.12")

In [ ]:
dirs[5]

## 15.36.55 skydip

In [ ]:
dataset0 = dirs[5]
a5 = qubicfp()
a5.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a5.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60 # 1 hora

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_mask = sat_mask
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
sat_idx

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15,9))

ax[0,0].plot(tt, todarray[0], color='grey')
ax[0,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0,0].axhline(y = max(todarray[0]), color= 'red', ls= '--', label='Saturation limit') 
ax[0,0].legend(fontsize=14, loc ="lower left")
ax[0,0].text(0.8,0.03, f"{sat_frac[0]*100:.2f}%", transform=ax[0, 0].transAxes, fontsize=16, color='green')
ax[0,0].set_title('TES 1')
#ax[0,0].grid()


ax[0,1].plot(tt, todarray[4], color='grey')
ax[0,1].axhline(y = min(todarray[4]), color= 'red', ls= '--', label='Saturation limit') 
ax[0,1].legend(fontsize=14)
ax[0,1].text(0.15,0.9, f"{sat_frac[4]*100:.1f}%", transform=ax[0, 1].transAxes, fontsize=16, color='green')
ax[0,1].set_title('TES 5')

#ax[0,1].grid()


ax[1,0].plot(tt, todarray[44], color='grey')
ax[1,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,0].set_xlabel('Time [sec]', fontsize=18)
ax[1,0].axhline(y = min(todarray[44]), color= 'red', ls= '--', label='Saturation limit') 
ax[1,0].legend(fontsize=14)
ax[1,0].text(0.03,0.9, f"{sat_frac[44]*100:.1f}%", transform=ax[1, 0].transAxes, fontsize=16, color='green')
ax[1,0].set_title('TES 45')

#ax[1,0].grid()


ax[1,1].plot(tt, todarray[249], color='grey')
ax[1,1].set_xlabel('Time [sec]', fontsize=18)
ax[1,1].axhline(y = min(todarray[249]), color= 'red', ls= '--', label='Saturation limit') 
ax[1,1].legend(fontsize=14, loc="upper right")
ax[1,1].text(0.03,0.9, f"{sat_frac[249]*100:.2f}%", transform=ax[1, 1].transAxes, fontsize=16, color='green')
ax[1,1].set_title('TES 250')
#ax[1,1].grid()



plt.savefig("TES_saturated2.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
thr = 2e5#, 3e5, 5e5]  # Multiple thresholds for better detection
window_size = 500  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
xc_7, xcf_7 = jump_data[7]['xc'], jump_data[7]['xcf']
xc_55, xcf_55 = jump_data[55]['xc'], jump_data[55]['xcf']
xc_254, xcf_254 = jump_data[254]['xc'], jump_data[254]['xcf']
xc_205, xcf_205 = jump_data[205]['xc'], jump_data[205]['xcf']


In [ ]:
haar_7 = fluxjumps.haar_function(todarray[7])
haar_55 = fluxjumps.haar_function(todarray[55])

In [ ]:
TES_yes

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 8))

ax[0,0].plot(tt, todarray[7], color='grey')
ax[0,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0,0].set_title('TES 8', fontsize=15)

ax[1,0].plot(tt, haar_7, color='red')
ax[1,0].set_ylabel('Haar filter', fontsize=18)
ax[1,0].set_xlabel('Time [sec]', fontsize=18)
ax[1,0].axhline(y=thr, ls='--', color='green', label='Threshold')
ax[1,0].legend(fontsize=15)
#ax[1,0].grid()

ax[0,1].plot(tt, todarray[55], color='grey')
ax[0,1].set_title('TES 56', fontsize=15)
#ax[0,1].grid()

ax[1,1].plot(tt, abs(haar_55), color='red')
ax[1,1].set_xlabel('Time [sec]', fontsize=18)
ax[1,1].axhline(y=thr, ls='--', color='green', label='Threshold')
ax[1,1].legend(fontsize=15)
#ax[1,1].grid()

plt.savefig("TES_haar2.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 10))

ax[0,0].plot(tt, todarray[7], color='grey')
ax[0,0].plot(tt[xc_7], todarray[7][xc_7], marker='o', lw=0, color='red', label='jump start')
ax[0,0].plot(tt[xcf_7], todarray[7][xcf_7], marker='s', lw=0, color='green', label='jump end')
ax[0,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0,0].legend(fontsize=15)
ax[0,0].set_title('TES 8', fontsize=15)


ax[0,1].plot(tt, todarray[55], color='gray')
ax[0,1].plot(tt[xc_55], todarray[55][xc_55], marker='o', lw=0, color='red', label='jump start')
ax[0,1].plot(tt[xcf_55], todarray[55][xcf_55], marker='s', lw=0, color='green', label='jump end')
#ax[0,1].set_ylabel('Flux [ADUs]', fontsize=18)
#ax[0,1].set_xlabel('Time [sec]', fontsize=18)
ax[0,1].legend(fontsize=15)
ax[0,1].set_title('TES 56', fontsize=15)


ax[1,0].plot(tt, todarray[205], color='gray')
ax[1,0].plot(tt[xc_205], todarray[205][xc_205], marker='o', lw=0, color='red', label='jump start')
ax[1,0].plot(tt[xcf_205], todarray[205][xcf_205], marker='s', lw=0, color='green', label='jump end')
ax[1,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,0].set_xlabel('Time [sec]', fontsize=18)
ax[1,0].legend(fontsize=15)
ax[1,0].set_title('TES 206', fontsize=15)


ax[1,1].plot(tt, todarray[254], color='gray')
ax[1,1].plot(tt[xc_254], todarray[254][xc_254], marker='o', lw=0, color='red', label='jump start')
ax[1,1].plot(tt[xcf_254], todarray[254][xcf_254], marker='s', lw=0, color='green', label='jump end')
#ax[1,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,1].set_xlabel('Time [sec]', fontsize=18)
ax[1,1].legend(fontsize=15)
ax[1,1].set_title('TES 255', fontsize=15)


plt.savefig("TES_detections2.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1000, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
xcdt_7, xcfdt_7 = np.array(dt_jump_data[7]['xcdt']), np.array(dt_jump_data[7]['xcfdt'])
xcdt_55, xcfdt_55 = np.array(dt_jump_data[55]['xcdt']), np.array(dt_jump_data[55]['xcfdt'])
xcdt_254, xcfdt_254 = np.array(dt_jump_data[254]['xcdt']), np.array(dt_jump_data[254]['xcfdt'])
xcdt_205, xcfdt_205 = np.array(dt_jump_data[205]['xcdt']), np.array(dt_jump_data[205]['xcfdt'])

In [ ]:
ypred7 = dt.define_model(tt, todarray[7], 3)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15, 5))

ax[0].plot(tt, todarray[7], color='grey')
ax[0].plot(tt, ypred7, 'm.', label='DT prediction')
ax[0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0].set_xlabel('Time [sec]', fontsize=18)
ax[0].legend(fontsize=15)
ax[0].set_title('TES 8', fontsize=15)

ax[1].plot(tt, todarray[7], color='gray')
for i in range(len(valini)):
    ax[1].axhline(y=valini[i], ls='--', color='purple')
    ax[1].axhline(y=valfin[i], ls='--', color='purple')
ax[1].plot(tt[xcdt_7], todarray[7][xcdt_7], marker='o', lw=0, color='red', label='jump start')
ax[1].plot(tt[xcfdt_7], todarray[7][xcfdt_7], marker='s', lw=0, color='green', label='jump end')
ax[1].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1].set_xlabel('Time [sec]', fontsize=18)
ax[1].legend(fontsize=15)
ax[1].set_title('TES 8', fontsize=15)


plt.savefig("TES_dt2.png")

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
todnew = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:            
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)
            todnew[idx] = corr.move_offset(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
#corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 10))

ax[0,0].plot(tt, todarray[7], color='grey')
ax[0,0].plot(tt[xc_7], todarray[7][xc_7], marker='o', lw=0, color='red', label='jump start')
ax[0,0].plot(tt[xcf_7], todarray[7][xcf_7], marker='s', lw=0, color='green', label='jump end')
ax[0,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0,0].legend(fontsize=15)
ax[0,0].set_title('TES 8', fontsize=15)

ax[1,0].plot(tt, todarray[55], color='gray')
ax[1,0].plot(tt[xc_55], todarray[55][xc_55], marker='o', lw=0, color='red', label='jump start')
ax[1,0].plot(tt[xcf_55], todarray[55][xcf_55], marker='s', lw=0, color='green', label='jump end')
ax[1,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,0].set_xlabel('Time [sec]', fontsize=18)
ax[1,0].legend(fontsize=15)
ax[1,0].set_title('TES 56', fontsize=15)


#ax[1,0].plot(tt, todarray[205], color='gray')
ax[0,1].plot(tt, todnew[7], color='gray', label='signal shifted')
#ax[0,1].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0,1].legend(fontsize=15, loc = 'lower right')
ax[0,1].set_title('TES 8', fontsize=15)


#ax[1,1].plot(tt, todarray[254], color='gray')
ax[1,1].plot(tt, todnew[55], color='gray', label='signal shifted')
ax[1,1].set_xlabel('Time [sec]', fontsize=18)
ax[1,1].legend(fontsize=15, loc = 'lower right')
ax[1,1].set_title('TES 56', fontsize=15)


plt.savefig("move_offset2.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
arr1_55 = np.array(xcdt_55)
arr2_55 = np.array(xcfdt_55)

arr1_55_c = np.array(xc_55)
arr2_55_c = np.array(xcf_55)

arr1_254 = np.array(xcdt_254)
arr2_254 = np.array(xcfdt_254)

arr1_254_c = np.array(xc_254)
arr2_254_c = np.array(xcf_254)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15, 8))


ax[0].plot(tt, corrected_tod[55], color='gray', label='Corrected signal')
ax[0].plot(tt, todarray[55], color='green', label='signal with flux jumps')
ax[0].set_xlabel('Time [sec]', fontsize=18)
ax[0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0].legend(fontsize=15)
ax[0].set_title('TES 56', fontsize=15)


ax[1].plot(tt, corrected_tod[55], color='gray', label='Corrected signal')
for i, (a1, a2) in enumerate(zip(arr1_55, arr2_55)):
    ax[1].plot(
        tt[a1:a2],
        corrected_tod[55][a1:a2],
        color='magenta', 
    label='Constrained realization' if i == 0 else "")

ax[1].set_xlabel('Time [sec]', fontsize=18)
ax[1].legend(fontsize=15)
ax[1].set_title('TES 56', fontsize=15)


plt.savefig("const_real2.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 10))

ax[0,0].plot(tt, corrected_tod_nodt[55], color='gray', label='Corrected signal, without DT')
for arr in arr1_55_c:
    ax[0,0].axvline(x=tt[arr], color='magenta', ls='--')

ax[0,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0,0].legend(fontsize=15, loc='lower right')
ax[0,0].set_title('TES 56', fontsize=15)


ax[0,1].plot(tt, corrected_tod[55], color='gray', label='Corrected signal, with DT')
for arr in arr1_55:
    ax[0,1].axvline(x=tt[arr], color='magenta', ls='--')
#ax[1].set_xlabel('Time [sec]', fontsize=18)
ax[0,1].legend(fontsize=15, loc='lower right')
ax[0,1].set_title('TES 56', fontsize=15)

ax[1,0].plot(tt, corrected_tod_nodt[254], color='gray', label='Corrected signal, without DT')
for arr in arr1_254_c:
    ax[1,0].axvline(x=tt[arr], color='magenta', ls='--')
ax[1,0].set_xlabel('Time [sec]', fontsize=18)
ax[1,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,0].legend(fontsize=15, loc='upper left')
ax[1,0].set_title('TES 255', fontsize=15)

#ax[1,0].set_title('without DT', fontsize=18)


ax[1,1].plot(tt, corrected_tod[254], color='gray', label='Corrected signal, with DT')
for arr in arr1_254:
    ax[1,1].axvline(x=tt[arr], color='magenta', ls='--')
ax[1,1].set_xlabel('Time [sec]', fontsize=18)
ax[1,1].legend(fontsize=15, loc='upper left')
ax[1,1].set_title('TES 255', fontsize=15)

#ax[1,1].set_title('with DT', fontsize=18)

plt.savefig("corr_comparison2.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 8))

ax[0,0].plot(tt, todarray[55], color='gray', label='original TOD')
#ax[0,0].plot(tt, corrected_tod_nodt[55], color='orange', label='without DT')
#ax[0,0].plot(tt, corrected_tod[55], color='green', label='with DT')
ax[0,0].set_ylabel('Flux [ADUs]', fontsize=18)
#ax[0,0].set_xlabel('Time [sec]', fontsize=18)
ax[0,0].legend(fontsize=11)


#ax[0,1].plot(tt, todarray[55], color='gray', label='original TOD')
ax[0,1].plot(tt, corrected_tod_nodt[55], color='orange', label='without DT')
ax[0,1].plot(tt, corrected_tod[55], color='green', label='with DT')

#ax[0,1].set_ylabel('Flux [ADUs]', fontsize=18)
#ax[0,1].set_xlabel('Time [sec]', fontsize=18)
ax[0,1].legend(fontsize=11)

ax[1,0].plot(tt, todarray[20], color='gray', label='original TOD')
#ax[1,0].plot(tt, corrected_tod_nodt[55], color='orange', label='without DT')
#ax[1,0].plot(tt, corrected_tod[55], color='green', label='with DT')
ax[1,0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,0].set_xlabel('Time [sec]', fontsize=18)
ax[1,0].legend(fontsize=11)

#ax[1,1].plot(tt, todarray[20], color='gray', label='original TOD')
ax[1,1].plot(tt, corrected_tod_nodt[20], color='orange', label='without DT')
ax[1,1].plot(tt, corrected_tod[20], color='green', label='with DT')

#ax[1,1].set_ylabel('Flux [ADUs]', fontsize=18)
ax[1,1].set_xlabel('Time [sec]', fontsize=18)
ax[1,1].legend(fontsize=11)

plt.savefig("TES_corr2_hcc.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 5))

ax[0].plot(tt, corrected_tod_nodt[254], color='orange', label='without DT')
ax[0].plot(tt, corrected_tod[254], color='green', label='with DT')
ax[0].plot(tt, todarray[254], color='gray', label='original TOD')

ax[0].set_xlabel('Time [sec]', fontsize=18)
ax[0].set_ylabel('Flux [ADUs]', fontsize=18)
ax[0].legend(fontsize=11)

ax[1].plot(tt, corrected_tod_nodt[205], color='orange', label='without DT')
ax[1].plot(tt, corrected_tod[205], color='green', label='with DT')
ax[1].plot(tt, todarray[205], color='gray', label='original TOD')

ax[1].legend(fontsize=11)
ax[1].set_xlabel('Time [sec]', fontsize=18)
#ax[1].set_ylabel('Flux [ADUs]', fontsize=18)


plt.savefig("TES_corr_hcc.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
TES_yes

In [ ]:
TES_with_dt_jumps

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
metrics_tod, _ = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=False, use_corrected=False
)

In [ ]:
for i in TES_with_dt_jumps:
    print("TES:", i, np.mean(np.abs(metrics_haar[i]['z_scores']))/np.mean(np.abs(metrics_dt[i]['z_scores'])))

In [ ]:
plt.plot(tt, corrected_tod_nodt[157], label='without DT')
plt.plot(tt, corrected_tod[157], label='with DT')

#plt.plot(tt, todarray[157])


plt.legend()
plt.title('TES 157')

In [ ]:
plt.plot(tt, corrected_tod[254], label='correction with DT')
plt.plot(tt, corrected_tod_nodt[254], label='correction without DT')

plt.plot(tt, todarray[254], label='TOD')


plt.legend()
plt.title('TES 254')

In [ ]:
metrics_haar[254]

In [ ]:
metrics_dt[254]

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=True)

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2805_15.36.55", dataset_name="2805_15.36.55")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,6))

for i in TES_with_dt_jumps:
    ax[0].plot(abs(np.asarray(offset_nodt[i])), marker='s', lw=0, label='TES_{}'.format(i+1))
    ax[0].legend(loc='upper right')
    
ax[0].axhspan(ymin=3.5e5, ymax=5.5e5, color='grey', alpha=0.3)
ax[0].axhline(y=4.5e5, color='black')
ax[0].axhline(y=9e5, color='black')

ax[0].set_ylim(2e5, 1.5e6)
ax[0].set_ylabel('Amplitude')
ax[0].set_xlabel('# jumps found')
ax[0].set_title('Detection without DT')

for i in TES_with_dt_jumps:
    ax[1].plot(abs(np.asarray(offset[i])), marker='s', lw=0, label='TES_{}'.format(i+1))
    ax[1].legend(loc='upper right')
    
ax[1].axhspan(ymin=3.5e5, ymax=5.5e5, color='grey', alpha=0.3)
ax[1].axhline(y=4.5e5, color='black')
ax[1].axhline(y=9e5, color='black')

ax[1].set_ylim(2e5, 1.5e6)
#ax[1].set_ylabel('Amplitude')
ax[1].set_xlabel('# jumps found')
ax[1].set_title('Detection with DT')
plt.savefig("TES_amplitude_hcc.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
off1 = []
for i in TES_with_dt_jumps:
    off1.append(abs(np.asarray(offset[i])))
    
off2 = []
for i in TES_with_dt_jumps:
    off2.append(abs(np.asarray(offset_nodt[i])))

In [ ]:
plt.hist(np.concatenate(off1), bins=15, range=(0.2e6, 0.8e6))
plt.hist(np.concatenate(off2), bins=15, range=(0.2e6, 0.8e6))


## 16.11.43 skydip

In [ ]:
dataset0 = dirs[6]
a6 = qubicfp()
a6.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a6.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60 # 1 hora

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
thr = 3e5  # Multiple thresholds for better detection
window_size = 500  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf  = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=3e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=False)

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=True)

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=False, use_corrected=True
)



In [ ]:
for i in TES_with_dt_jumps:
    print("TES:", i, np.mean(np.abs(metrics_haar[i]['z_scores']))/np.mean(np.abs(metrics_dt[i]['z_scores'])))

In [ ]:
metrics_haar[77]

In [ ]:
metrics_dt[77]

In [ ]:
plt.plot(tt, corrected_tod[210])
plt.plot(tt, corrected_tod_nodt[210])
plt.plot(tt, todarray[210])


In [ ]:


plt.plot(tt, todarray[77])
#plt.plot(tt, corrected_tod[77], label='with DT')
plt.plot(tt, corrected_tod_nodt[77], label='without DT')

plt.legend()

In [ ]:
plt.plot(tt, corrected_tod_nodt[51], label='without DT')

plt.plot(tt, corrected_tod[51], label='with DT')
plt.plot(tt, todarray[51])
plt.legend()

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2805_16.11.43", dataset_name="2805_16.11.43")